# Лабораторная 5. Что произойдёт в плохой день

Четыре предыдущие работы превратились в четыре тетради, которые надо
запускать по порядку каждое утро. В понедельник вы запускаете их сами.
Во вторник вы в отпуске, запускает коллега – и пропускает третью работу,
потому что «там всё равно ничего не меняется». В среду источник отдал половину
файла, расчёт отработал без ошибок, витрина обновилась наполовину, дашборд
показал провал продаж, и в одиннадцать начинается совещание, на котором вы
объясняете, что продажи в порядке, а не в порядке – половина файла.

В хороший день такая обработка работает прекрасно. Вопрос в том, что
происходит в плохой: когда источник недоступен, когда расчёт упал на середине,
когда данные за вчера приехали только сегодня к обеду и когда кто-то нажал
запуск дважды. Сегодняшняя работа – про это.

## Перед началом: новые слова

| Термин | Простыми словами |
|---|---|
| **Задача – task** | Один шаг обработки, который можно запустить и перезапустить отдельно. |
| **Зависимость – dependency** | Правило «этот шаг не начнётся, пока не закончился тот». |
| **Граф задач – DAG** | Схема шагов и зависимостей без циклов: по ней видно, что за чем идёт. |
| **Оркестрация – orchestration** | Управление запуском: порядок, расписание, повторы, реакция на падение. |
| **Повторная попытка – retry** | Автоматический повтор шага после сбоя, который может пройти сам. |
| **Инкрементальная обработка – incremental processing** | Пересчёт только новой порции данных вместо полного пересчёта. |
| **Пересчёт истории – backfill** | Повторный расчёт за уже прошедший период: нашли ошибку или данные приехали позже. |
| **Происхождение данных – data lineage** | Ответ на вопрос «из чего получена эта таблица и что сломается, если её изменить». |

## Подготовка

Ячейка подключает минимальный оркестратор курса и слой CLEAN из третьей работы.

In [ ]:
# Готовим окружение: в VS Code и Jupyter берём открытую папку курса,
# в Colab материалы скачиваются один раз. Всё остальное – в src/notebook_env.py.
from pathlib import Path
import subprocess, sys

ROOT = next((c for s in (Path.cwd().resolve(), Path("/content/data-engineering-ai-course"))
             for c in (s, *s.parents) if (c / ".course-root").exists()), None)
if ROOT is None:                                   # чистая сессия Colab
    url = ''
    if not url:
        raise RuntimeError("Откройте папку курса в VS Code/Jupyter или склонируйте "
                           "репозиторий в /content/data-engineering-ai-course")
    subprocess.run(["git", "clone", "--depth", "1", url,
                    "/content/data-engineering-ai-course"], check=True)
    ROOT = Path("/content/data-engineering-ai-course")
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
from notebook_env import setup_course
setup_course(ROOT)


In [ ]:
from datetime import date


import duckdb
import pandas as pd
import paths
from dq import CRITICAL, DataQualityError, Suite
from lab_utils import require_artifact
from pipeline import Pipeline, Task
from stages import ensure_stage

pd.set_option("display.width", 170)

# В отдельной Colab-сессии можно восстановить уже пройденный CLEAN-слой.
ensure_stage("clean")
for t in ["clean_orders", "clean_items"]:
    require_artifact(paths.CLEAN / f"{t}.parquet", "ЛР 3")

con = duckdb.connect(str(paths.WAREHOUSE))
con.execute("CREATE SCHEMA IF NOT EXISTS core")
con.execute("CREATE SCHEMA IF NOT EXISTS mart")
print("хранилище:", paths.WAREHOUSE.relative_to(paths.ROOT))

In [ ]:
from course_visuals import show_external_visual, show_visual

show_visual("dag")

## Из чего состоит обработка

> **Task (задача)** – минимальная единица работы, которую можно запустить,
> перезапустить и у которой есть понятный результат: «загрузить файл»,
> «проверить качество», «пересобрать витрину».
>
> **Dependency (зависимость)** – утверждение «задача B не может начаться,
> пока не закончилась A».
>
> **DAG (directed acyclic graph)** – направленный ациклический граф задач.
> Ациклический означает буквально: если A зависит от B, а B от A, система
> не сможет решить, с чего начать, – такой граф невозможно выполнить.

Зачем разбивать на задачи, если можно написать один длинный скрипт:

* **перезапуск** – упало на шаге 4, не надо повторять шаги 1–3;
* **наблюдаемость** – видно, какой именно шаг падает и сколько он длится;
* **параллельность** – независимые ветки можно считать одновременно;
* **повторное использование** – «проверить качество» нужна в трёх местах;
* **понятность** – DAG читается как схема, а скрипт на 500 строк – нет.

> **Orchestration (оркестрация)** – управление выполнением задач: порядок,
> расписание, повторные попытки, реакция на падения, уведомления.
>
> **Scheduler (планировщик)** – часть оркестратора, которая решает, что пора
> запускать. Простейший планировщик – `cron`.

In [ ]:
CLEAN_TABLES = ["clean_orders", "clean_items", "clean_payments", "clean_reviews",
                "clean_customers", "clean_products", "clean_sellers"]


def t_prepare(ctx: dict) -> str:
    """Имитация чистого окружения: сносим таблицы, которые обработка создаёт сама."""
    for t in CLEAN_TABLES:
        ctx["con"].execute(f"DROP TABLE IF EXISTS {t}")
    ctx["con"].execute("DROP TABLE IF EXISTS core.fact_order_items")
    return "окружение очищено"


def t_ingest(ctx: dict) -> int:
    """Загружаем проверенный слой в хранилище. CREATE OR REPLACE – шаг повторяем."""
    for t in CLEAN_TABLES:
        src = (paths.CLEAN / f"{t}.parquet").as_posix()
        ctx["con"].execute(
            f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM read_parquet('{src}')")
    return len(CLEAN_TABLES)


def t_validate(ctx: dict) -> dict:
    """Контроль качества: нарушение критического правила роняет шаг."""
    df = ctx["con"].execute("SELECT * FROM clean_orders").df()
    report = (Suite("clean_orders")
              .not_null("order_id").unique("order_id")
              .not_null("order_purchase_timestamp")
              .custom("статус delivered ⇒ есть дата доставки",
                      lambda d: d["is_delivered"]
                      & d["order_delivered_customer_date"].isna())
              .run(df))
    report.raise_if_critical()
    return {"n_rows": report.n_rows, "passed": report.passed}


def t_transform(ctx: dict) -> int:
    """Пересборка таблицы фактов."""
    if ctx.get("break_transform"):
        raise RuntimeError("источник отдал неполные данные (имитация аварии)")
    ctx["con"].execute("""
        CREATE OR REPLACE TABLE core.fact_order_items AS
        SELECT i.order_id, i.order_item_id,
               CAST(strftime(CAST(o.order_purchase_timestamp AS DATE), '%Y%m%d') AS INTEGER)
                   AS date_key,
               CAST(o.order_purchase_timestamp AS DATE) AS order_date,
               o.customer_id, i.product_id, i.seller_id,
               i.price, i.freight_value,
               o.order_status, o.is_delivered, o.is_late, o.delivery_days
        FROM clean_items i JOIN clean_orders o USING (order_id)
    """)
    return ctx["con"].execute(
        "SELECT COUNT(*) FROM core.fact_order_items").fetchone()[0]


print("шаги описаны")
#

## Что происходит, когда обработка падает на середине

Падения – не исключительная ситуация, а обычный режим работы. Источник
недоступен, диск кончился, партнёр прислал файл в другой кодировке,
у кластера отобрали ресурсы.

Важно различать два типа ошибок:

| Тип | Пример | Правильная реакция |
|-----|--------|--------------------|
| **Временная (transient)** | таймаут сети, блокировка таблицы | повторить попытку (retries) |
| **Постоянная (permanent)** | нет колонки, нарушен контракт данных | повторять бессмысленно – падать и звать человека |

`retries` лечит только первый тип. Три повторные попытки прочитать файл
со сломанной схемой – это просто три одинаковые ошибки и потерянные минуты.

И главный вопрос этой работы: **что осталось в данных после падения?**
Если шаг 3 успел переписать таблицу фактов, а шаг 4 упал, то в хранилище
теперь новый факт и старая витрина. Это называется **частичным состоянием**,
и именно из-за него нельзя просто «запустить ещё раз и посмотреть».

In [ ]:
# Обработка с неустойчивым источником и с настоящей аварией.
attempts = {"flaky": 0}


def t_flaky_source(ctx: dict) -> str:
    attempts["flaky"] += 1
    if attempts["flaky"] < 3:
        raise ConnectionError("источник не ответил (таймаут)")
    return "данные получены с третьей попытки"


def t_broken_transform(ctx: dict):
    raise ValueError("в выгрузке нет колонки price")


def t_should_not_run(ctx: dict) -> str:
    return "эта задача не должна запуститься"


demo = Pipeline("demo_failures", [
    Task("extract", t_flaky_source, retries=3, retry_delay=0.1),
    Task("transform", t_broken_transform, depends_on=["extract"]),
    Task("build_mart", t_should_not_run, depends_on=["transform"]),
], log_dir=paths.LOGS, runs_dir=paths.RUNS)

print(demo.draw(), "\n")
demo.quiet().run({"con": con}).show()

Три вещи, которые здесь видно:

1. `extract` упал дважды и выполнился с третьей попытки – retries сделали
   своё дело, человека будить не пришлось;
2. `transform` упал по-настоящему: отсутствующая колонка повтором не лечится;
3. `build_mart` **не запустился вообще** и получил статус `upstream_failed`.
   Это принципиально: лучше не обновить витрину, чем обновить её мусором.

## Повторный запуск не должен удвоить март

> **Идемпотентность** – свойство операции давать один и тот же результат
> независимо от того, выполнили её один раз или пять.

Это не академическая тонкость, а ежедневная практика: обработку перезапускают
после падений, после исправления кода, после позднего приезда данных.
Если шаг не идемпотентен, каждый перезапуск портит данные.

Типичные способы сделать шаг идемпотентным:

| Приём | Как выглядит | Когда подходит |
|-------|--------------|----------------|
| Полная перезапись | `CREATE OR REPLACE TABLE ...` | таблица небольшая, пересчёт дешёвый |
| Удалить-и-вставить период | `DELETE WHERE date BETWEEN ...; INSERT ...` | инкрементальная загрузка по периодам |
| Перезапись партиции | перезаписать файл `date=2018-03-17/` | данные в файлах, партиционированы |
| UPSERT / MERGE по ключу | `MERGE INTO ... ON key` | нужны обновления отдельных строк |
| Дедупликация на чтении | `ROW_NUMBER() OVER (PARTITION BY key ORDER BY ts DESC) = 1` | источник шлёт дубликаты |

Чего делать **нельзя**: слепой `INSERT INTO` (или `to_csv(mode='a')`) без
удаления старых данных за тот же период. Именно так ломается большинство
первых конвейеров.

In [ ]:
# Смотрим на проблему вживую: неидемпотентная загрузка витрины.
con.execute("""
    CREATE OR REPLACE TABLE mart.demo_sales (order_date DATE, revenue DOUBLE)
""")


def load_not_idempotent(con, d1: str, d2: str) -> None:
    con.execute(f"""
        INSERT INTO mart.demo_sales
        SELECT order_date, SUM(price) FROM core.fact_order_items
        WHERE order_date BETWEEN DATE '{d1}' AND DATE '{d2}'
        GROUP BY order_date
    """)


t_ingest({"con": con})
t_transform({"con": con})

for run in (1, 2, 3):
    load_not_idempotent(con, "2018-01-01", "2018-01-07")
    stats = con.execute("""
        SELECT COUNT(*) AS строк, ROUND(SUM(revenue), 2) AS выручка
        FROM mart.demo_sales
    """).df()
    print(f"после запуска {run}: {stats.to_dict('records')[0]}")

print("\nДанные не «обновились» – они накопились. Дашборд показывает"
      "\nвыручку, которой не было, и найти это можно только глазами.")

## Как пересчитать прошлый период и не тронуть остальные

> **Incremental processing (инкрементальная обработка)** – пересчёт только
> новой или изменившейся порции данных вместо полного пересчёта.
>
> **Backfill** – пересчёт задним числом за уже прошедший период: нашли ошибку
> в логике, поздно приехали данные, добавили новую колонку в витрину.

Эти две вещи связаны напрямую: **инкрементальная обработка полезна только
тогда, когда шаг идемпотентен**. Иначе backfill за март добавит вторую копию
марта вместо того, чтобы его исправить.

Отсюда практическое правило: шаг обработки принимает **период данных**
(а не «всё, что есть») и умеет перезаписывать этот период целиком.

Ещё одна частая путаница: **дата фактического старта ≠ период данных**.
У DAG Run есть `logical_date` – логический идентификатор запуска; сам по себе
он не говорит, какие строки нужно читать. Для выбора данных Airflow даёт
`data_interval_start` и `data_interval_end`. Ежедневный запуск в 03:00 17
марта часто считает 16 марта, но точные границы задаёт расписание и timetable.

## Собираем витрину продаж по дням

Витрина `mart.daily_sales` – **одна строка на день**:

| колонка | смысл |
|---------|-------|
| `order_date` | дата покупки |
| `n_orders` | число различных заказов за день |
| `n_items` | число позиций |
| `revenue` | сумма `price` |
| `freight` | сумма `freight_value` |

Витрина строится **за период** `[ctx["start"], ctx["end"]]` – так шаг можно
запускать и на весь исторический диапазон, и на один день, и на один месяц
при backfill.

### Задание 1. Сделайте шаг `build_mart` идемпотентным

Ниже – реализация, которая просто дописывает строки. Перепишите её так, чтобы:

1. повторный запуск за тот же период **не менял** содержимое витрины;
2. запуск за период, который уже посчитан, **пересчитывал** его (это нужно
   для backfill, когда данные за март изменились);
3. данные **за другие периоды не терялись** – запуск за март не имеет права
   удалить февраль.

Третье требование важно: самый простой способ «сделать идемпотентно» –
пересобрать всю таблицу целиком – здесь не подходит, потому что шаг получает
только свой период. Проверка это поймает.

In [ ]:
def t_build_mart(ctx: dict) -> int:
    """Собирает mart.daily_sales за период [ctx['start'], ctx['end']]."""
    con = ctx["con"]
    start, end = ctx["start"], ctx["end"]

    con.execute("""
        CREATE TABLE IF NOT EXISTS mart.daily_sales (
            order_date DATE, n_orders BIGINT, n_items BIGINT,
            revenue DOUBLE, freight DOUBLE
        )
    """)

    # TODO: сделайте этот шаг идемпотентным (см. три требования выше).
    con.execute(f"""
        INSERT INTO mart.daily_sales
        SELECT order_date,
               COUNT(DISTINCT order_id) AS n_orders,
               COUNT(*)                 AS n_items,
               SUM(price)               AS revenue,
               SUM(freight_value)       AS freight
        FROM core.fact_order_items
        WHERE order_date BETWEEN DATE '{start}' AND DATE '{end}'
        GROUP BY order_date
    """)

    return con.execute(f"""
        SELECT COUNT(*) FROM mart.daily_sales
        WHERE order_date BETWEEN DATE '{start}' AND DATE '{end}'
    """).fetchone()[0]


def t_check(ctx: dict) -> dict:
    """Финальная проверка витрины: pipeline обязан сам замечать поломку."""
    con = ctx["con"]
    dup = con.execute("""
        SELECT COUNT(*) FROM (
            SELECT order_date FROM mart.daily_sales
            GROUP BY order_date HAVING COUNT(*) > 1)
    """).fetchone()[0]
    if dup:
        raise ValueError(f"в витрине {dup} дублирующихся дат – шаг не идемпотентен")
    neg = con.execute(
        "SELECT COUNT(*) FROM mart.daily_sales WHERE revenue < 0").fetchone()[0]
    if neg:
        raise ValueError(f"отрицательная выручка в {neg} днях")
    return {"days": con.execute(
        "SELECT COUNT(*) FROM mart.daily_sales").fetchone()[0]}


print("шаги build_mart и check описаны")

### Задание 2. Соберите DAG

Ниже собрана обработка, но у половины задач **не указаны зависимости**.
Расставьте их так, чтобы:

* шаги выполнялись в осмысленном порядке;
* ни один шаг не начинался раньше, чем появились нужные ему данные;
* при падении проверки качества витрина **не пересобиралась**.

Подсказка: задача `prepare` специально удаляет таблицы, имитируя чистое
окружение. Если зависимости расставлены неверно, обработка не просто
«сделает странно» – она упадёт, потому что нужных таблиц ещё не существует.

In [ ]:
pipe = Pipeline("olist_daily", [
    Task("prepare",    t_prepare),
    Task("ingest",     t_ingest,     depends_on=["prepare"], retries=2, retry_delay=0.2),
    # TODO: расставьте depends_on у оставшихся задач
    Task("validate",   t_validate,   depends_on=[]),
    Task("transform",  t_transform,  depends_on=[]),
    Task("build_mart", t_build_mart, depends_on=[]),
    Task("check",      t_check,      depends_on=[]),
], log_dir=paths.LOGS, runs_dir=paths.RUNS)

print(pipe.draw())

In [ ]:
# Полный прогон за всю историю.
bounds = con.execute("""
    SELECT MIN(CAST(order_purchase_timestamp AS DATE)),
           MAX(CAST(order_purchase_timestamp AS DATE)) FROM clean_orders
""").fetchone()
FULL = {"start": str(bounds[0]), "end": str(bounds[1])}
print("период данных:", FULL)

con.execute("DROP TABLE IF EXISTS mart.daily_sales")
result = pipe.quiet().run({"con": con, **FULL}).show()

In [ ]:
from checks.lab05 import (check_task_1, check_task_2, check_task_3, lab05_result)

check_task_2(pipe, result)

In [ ]:
check_task_1(pipe, con, FULL)

### Задание 3. Backfill

Ситуация: выяснилось, что данные за **март 2018** приехали неполными,
источник их перезалил. Нужно пересчитать витрину только за март, не трогая
остальные месяцы и не пересобирая всё с нуля.

Заполните параметры запуска: период данных и список задач, которые
действительно нужно выполнить при backfill.

Подумайте: нужно ли при backfill заново выполнять `prepare` и `ingest`?
А `validate`? Ответ зависит от того, что именно изменилось – обоснуйте
свой выбор в ячейке ниже.

In [ ]:
BACKFILL = {
    "start": "____",          # TODO: первый день периода backfill
    "end": "____",            # TODO: последний день периода backfill
}
BACKFILL_TASKS = []           # TODO: какие задачи выполняем при backfill

pipe.run({"con": con, **BACKFILL}, only=BACKFILL_TASKS).show()

**Ваш ответ:** почему при backfill вы выполняете именно эти шаги?
В каком случае пришлось бы запускать `ingest` и `validate` тоже?

*(2–4 предложения)*

In [ ]:
check_task_3(pipe, con, FULL, BACKFILL, BACKFILL_TASKS)

## Что ломается в этом месте

**Обработка упала после преобразования.**

In [ ]:
before = con.execute("SELECT COUNT(*), ROUND(SUM(revenue), 2) FROM mart.daily_sales").fetchone()
broken = pipe.run({"con": con, **FULL, "break_transform": True})
after = con.execute("SELECT COUNT(*), ROUND(SUM(revenue), 2) FROM mart.daily_sales").fetchone()

print(f"\nвитрина до аварии : {before}")
print(f"витрина после     : {after}")
print(f"статусы задач     : "
      f"{ {k: v.status for k, v in broken.tasks.items()} }")
print("\nВитрина осталась прежней – её просто не тронули. Это правильное")
print("поведение: лучше вчерашние данные, чем сегодняшние наполовину.")
print("А вот таблица clean_* уже удалена и пересоздана шагом prepare/ingest –")
print("вот оно, частичное состояние: часть системы обновлена, часть нет.")

**Запустили исправную обработку дважды подряд.**

In [ ]:
pipe.run({"con": con, **FULL})
snapshot1 = con.execute(
    "SELECT COUNT(*), ROUND(SUM(revenue), 2), ROUND(SUM(n_orders), 0) FROM mart.daily_sales"
).fetchone()
pipe.run({"con": con, **FULL})
snapshot2 = con.execute(
    "SELECT COUNT(*), ROUND(SUM(revenue), 2), ROUND(SUM(n_orders), 0) FROM mart.daily_sales"
).fetchone()

print(f"\nпосле первого запуска : {snapshot1}")
print(f"после второго запуска : {snapshot2}")
print(f"идемпотентно: {snapshot1 == snapshot2}")

**Смотрим, что известно о вчерашнем запуске.**

In [ ]:
import json

meta = json.loads((paths.RUNS / "olist_daily_latest.json").read_text(encoding="utf-8"))
print("метаданные последнего запуска:")
print(json.dumps({k: meta[k] for k in ["pipeline", "run_id", "status", "seconds"]},
                 ensure_ascii=False, indent=1))
print("\nпо задачам:")
for name, t in meta["tasks"].items():
    print(f"  {name:<12} {t['status']:<16} {t['seconds']:>6.2f} c  попыток: {t['attempts']}")

log_file = paths.LOGS / "olist_daily.log"
print(f"\nпоследние строки лога ({log_file.relative_to(paths.ROOT)}):")
print("\n".join(log_file.read_text(encoding="utf-8").splitlines()[-6:]))

## Как узнать, что ночной запуск не состоялся

Четыре слова, которые часто произносят вместе и редко объясняют.

> **Metadata (метаданные)** – данные о данных и о процессах: когда обработка
> запускалась, сколько строк обработала, сколько заняла и чем закончилась.
> По ним отвечают на вопрос «а что было ночью», не гадая.
>
> **Lineage (происхождение данных)** – граф «какая таблица из какой получена».
> Нужен, чтобы ответить на два вопроса: «откуда взялось это число в дашборде»
> и «что сломается, если я изменю вот эту колонку».
>
> **Monitoring (мониторинг)** – наблюдение за системой и оповещение людей.
> Минимально полезный набор: обработка упала; обработка не запустилась вовсе;
> данных приехало подозрительно мало; проверки качества не прошли.
>
> **Reproducibility (воспроизводимость)** – способность получить тот же
> результат повторно. Требует трёх вещей: неизменяемого RAW,
> версионированного кода и идемпотентных шагов.

Отдельно про «обработка не запустилась вовсе»: это самый коварный отказ.
Упавшую обработку видно, а не стартовавшую – нет: дашборд молча показывает
вчерашние данные, и никто ничего не замечает до совещания.

In [ ]:
# Простейший lineage нашей системы – его можно построить прямо из зависимостей задач.
print(pipe.draw())
print("""
А вот lineage на уровне данных (что из чего получено):

  raw/*.csv → staging/parquet/* → clean/*.parquet → clean_* (DuckDB)
                                                       ↓
                                             core.fact_order_items
                                                       ↓
                                              mart.daily_sales → BI
""")

## Как это выглядело бы в Airflow

Тот же самый DAG, те же самые функции – меняется только обвязка:

```python
from airflow.sdk import dag, task
import pendulum

@dag(
    dag_id="olist_daily", schedule="0 3 * * *",
    start_date=pendulum.datetime(2017, 1, 1, tz="UTC"), catchup=True,
    default_args={"retries": 2, "retry_delay": pendulum.duration(minutes=5)},
    tags=["olist", "daily"],
)
def olist_daily():
    @task
    def ingest() -> str:
        return t_ingest()       # в жизни: URI/метаданные, не большой DataFrame

    @task
    def validate(raw_uri: str) -> str:
        return t_validate(raw_uri)

    raw_uri = ingest()
    clean_uri = validate(raw_uri)
    t_transform(clean_uri)      # остальные задачи оформляются так же

olist_daily()
```

Наш мини-оркестратор умещается в один процесс: он же читает описание задач,
он же запускает, он же пишет журнал. У настоящего оркестратора эти роли
разнесены, и это видно на схеме из документации:

In [ ]:
show_external_visual(
    "airflow_architecture",
    caption="Файлы DAG, планировщик, интерфейс и база метаданных – четыре разные части.")

Слева человек пишет файлы DAG – обычный код на Python, который лежит в папке.
Планировщик (Scheduler) читает эти файлы и решает, что пора запускать. Итог
каждого запуска пишется в базу метаданных (Metadata DB), и оттуда же его
показывает веб-интерфейс. Наши логи и файлы в `data/_runs` – это ровно она,
только очень простая.

Что Airflow добавляет к нашему мини-оркестратору:

| Возможность | Зачем |
|-------------|-------|
| планировщик и `catchup` | запускать по расписанию и автоматически догонять пропущенные дни |
| веб-интерфейс | видно историю запусков, логи каждой задачи, где именно упало |
| `data_interval_start/end` | передать задаче период данных, а не время старта |
| пулы и очереди | ограничить нагрузку на источник, распараллелить независимое |
| сенсоры | «подождать, пока появится файл» |
| алерты | письмо или сообщение в мессенджер при падении |

**Когда Airflow не нужен:** один скрипт, запускаемый раз в сутки, прекрасно
живёт в `cron` с логом и письмом при ненулевом коде возврата. Airflow
окупается, когда задач десятки, между ними есть зависимости и за ними
нужно наблюдать.

## Та же задача в другой системе

Прокатный стан. На печах и линии стоят датчики: температура, скорость,
вибрация – суммарно около 2000 сигналов, каждый пишется раз в секунду.
Данные попадают в локальный сервер цеха, оттуда раз в 15 минут выгружаются
в корпоративную сеть.

Что нужно бизнесу:

* оператору у стана – текущие показания и алерт при выходе за пределы
  (задержка не больше пары секунд);
* технологам – суточные отчёты по браку и по режимам;
* инженерам по надёжности – история за 3 года для анализа отказов;
* модель предсказания отказа подшипника – обучается раз в месяц.

Особенности: связь между цехом и дата-центром иногда пропадает на несколько
часов, и тогда данные приезжают пачкой задним числом. Время на датчиках
иногда «уплывает».

Ответьте письменно:

1. Какие здесь источники и какие два (как минимум) разных пути данных?
2. Где нужна потоковая обработка, а где достаточно пакетной?
3. Как хранить 3 года данных с 2000 сигналов в секунду? Оцените объём
   и предложите формат и партиционирование.
4. Что произойдёт с суточным отчётом, если данные за вчера приехали сегодня
   в 14:00? Как это обработать?
5. Какие шаги обработки должны быть идемпотентными и почему это здесь
   критично?
6. Что мониторить в первую очередь?
7. Какие компоненты в этой задаче лишние?

### Ваш ответ

1. Источники и пути данных: …
2. Потоковая и пакетная обработка: …
3. Хранение и партиционирование: …
4. Поздние данные: …
5. Идемпотентность: …
6. Мониторинг: …
7. Лишние компоненты: …

In [ ]:
lab05_result()
con.close()

## Вопросы для проверки себя

1. Ваша обработка упала на шаге `build_mart`. Коллега предлагает «просто
   запустить всё заново». В каком случае это безопасно, а в каком –
   испортит данные?
2. Почему `retries=5` не спасает от ошибки «в файле нет колонки `price`»?
   Как система должна реагировать на такую ошибку?
3. Чем отличается дата запуска обработки от периода данных, который она
   обрабатывает? Приведите пример, когда путаница между ними приводит
   к неверному отчёту.
4. Назовите три способа сделать шаг загрузки идемпотентным и укажите,
   в какой ситуации каждый из них предпочтительнее.
5. Что такое «обработка не запустилась вообще» и почему этот отказ опаснее
   падения? Как его обнаруживать?

## Что дальше

Обработка знает свои зависимости, пишет журнал, умеет повторить временный сбой
и не удваивает март после перезапуска. Но работает она на одной машине и по
расписанию. В шестой работе разберём, что меняется, когда данных становится
больше, события нужны через секунды, а результат ждёт модель.